# day-34-monitor-and-retrain — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt.

### Exercise 2 — PSI bin sensitivity

In [1]:
import numpy as np
rng = np.random.default_rng(1)
ref = rng.normal(0, 1, 3000); cur = rng.normal(0.7, 1.2, 1500)
def psi(reference, current, bins=10):
    r = np.asarray(reference, float); c = np.asarray(current, float)
    e = np.quantile(r, np.linspace(0, 1, bins + 1)); e[0], e[-1] = -np.inf, np.inf
    rp = np.clip(np.histogram(r, e)[0] / len(r), 1e-6, None)
    cp = np.clip(np.histogram(c, e)[0] / len(c), 1e-6, None)
    return float(np.sum((cp - rp) * np.log(cp / rp)))
for b in (5, 10, 20, 50):
    print(f"bins={b:2d}  PSI={psi(ref, cur, b):.3f}")
print("\nverdict (>0.25) is stable across bin counts here; with tiny current samples,")
print("too many bins -> near-empty bins -> PSI inflates and gets noisy.")

bins= 5  PSI=0.330
bins=10  PSI=0.389
bins=20  PSI=0.420
bins=50  PSI=0.464

verdict (>0.25) is stable across bin counts here; with tiny current samples,
too many bins -> near-empty bins -> PSI inflates and gets noisy.


### Exercise 6 — poisoned self-training loop

In [2]:
import numpy as np
rng = np.random.default_rng(0)
def generation(prev_quality, human_check):
    # new few-shots drawn from prior outputs; 10% carry an error
    err_rate = 0.10 * (0.15 if human_check else 1.0)   # human review catches most
    drift = -err_rate * 0.6 + rng.normal(0, 0.01)
    return max(0.0, prev_quality + drift)
for label, hc in [("no human check", False), ("judge + human sample", True)]:
    q = 0.86; hist = [q]
    for _ in range(5):
        q = generation(q, hc); hist.append(round(q, 3))
    print(f"{label:24s}: {hist}")
print("\nUnchecked: eval score decays each generation (error compounding).")
print("With a judge + human sample on collected cases: roughly stable.")

no human check          : [0.86, 0.801, 0.74, 0.686, 0.627, 0.562]
judge + human sample    : [0.86, 0.855, 0.859, 0.859, 0.843, 0.821]

Unchecked: eval score decays each generation (error compounding).
With a judge + human sample on collected cases: roughly stable.


### Exercises 1, 3, 4, 5 — sketches

**1. Multi-window burn-rate.** Keep two rolling windows (5-min, 1-hr). Page only if
`grounded_rate < floor` in **both**. A 2-min blip never fills the 5-min window enough to
breach; a 90-min outage breaches both → one page, no flapping.

**3. Concept drift.** Generate a stream where `topic` distribution and embeddings are fixed,
but on day 15 the *labelled correct answer* for "what is the refund window?" changes 30→14
days. PSI/KS on inputs ≈ 0. A re-labelled eval set (graded against the new policy) drops
sharply, and `thumbs_down_rate` rises — those are the only signals that move.

**4. Retrieval-age drift.** Add `retrieved_doc_age_days` per event. Track PSI of that
distribution vs a healthy reference; when the mass shifts old (corpus not being refreshed),
PSI climbs. Add it as a 5th `RefreshTrigger` reason with its own threshold.

**5. Trigger cooldown.** `RefreshTrigger` keeps a per-reason counter; a reason only "counts"
if it has held for `N` consecutive windows. Also store `last_refresh_ts` and require
`now - last_refresh_ts > min_interval`. Re-run §5 with a one-day news spike (PSI up for 1
window only) → reason never reaches `N` → no refresh.

### Answer key

1. Concept drift or retrieval decay — e.g. two-thirds of traffic starts asking about a product
   the corpus never ingested. Latency and error rate stay flat; **grounded-answer rate** (and
   thumbs-down) move.
2. PSI measures how much a distribution has shifted between a reference and a current sample
   (summed weighted log-ratio over bins). ~0.1–0.25 = moderate shift, >0.25 = significant.
3. Both compare only the *input* distribution. Concept drift changes P(answer | input) while
   the inputs look identical, so the input distributions match and PSI/KS see nothing.
4. Latency, traffic, errors, cost, quality (retrieval hit / grounded / judge score), feedback
   (thumbs-down / escalation). Infra dashboards usually miss **quality** and **feedback**.
5. To avoid twitchiness — any single signal is noisy (a news-driven PSI spike, a bad sample of
   feedback). Requiring 2+ independent reasons (plus a cooldown) means a real degradation, not
   noise, before you rebuild and re-deploy.
6. The model's errors and stylistic quirks get amplified each generation (error compounding,
   mode collapse, lost diversity). Mitigate by using human-labelled or human-corrected cases,
   and gating any model-generated training data through a judge plus a human-reviewed sample.
7. serve → emit trace → aggregate golden signals → detect drift / eval decay / feedback →
   `RefreshTrigger` (2+ reasons) → collect recent good cases → rebuild (re-index / re-tune /
   refresh few-shots) → **eval gate** (Day 33) → **canary rollout** (Day 33) → promote →
   candidate scores become the new baseline → back to serve.